# EEG · Live BCI Prediction Demo

Loads the trained LightGBM model saved by `eeg_lgb_focused.ipynb`, replays one
trial sample-by-sample and renders an `.mp4` that simulates real-time inference.

**Layout**
- **4 × 4 panels** — the 16 EEG channels (4 s scrolling window, CORAL when detecting)
- **Bottom panel** — smoothed LightGBM probability + button-press events (30 s scrolling)

Run `lgb-save` in `eeg_lgb_focused.ipynb` first to generate `models/lgb_demo.pkl`.

In [1]:
import os, sys, pickle
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")   # off-screen backend for FuncAnimation.save()
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from scipy.signal import welch

sys.path.insert(0, "..")
from src.preprocessing import load_trial

DARK, CARD, EDGE = "#0a0e17", "#111827", "#1f2937"
TEAL, CORAL, GOLD, WHITE = "#00e5cc", "#ff4f5e", "#ffc947", "#f0f4ff"

print("Imports OK")

Imports OK


In [2]:
with open("models/lgb_demo.pkl", "rb") as f:
    art = pickle.load(f)

model      = art["model"]
W          = art["W"]           # (C, C) EA whitening matrix
T_OPT      = art["T_OPT"]
FS         = art["FS"]
DECIM      = art["DECIM"]
FS_EFF     = art["FS_EFF"]
LP         = art["LP"]
HP         = art["HP"]
HORIZON    = art["HORIZON"]
SMOOTH_WIN = art["SMOOTH_WIN"]
thresh     = art["thresh"]
EEG_COLS   = art["EEG_COLS"]
SUBJECT    = art["SUBJECT"]
TACHE      = art["TACHE"]
DATA_PATH  = art["DATA_PATH"]
DEMO_TRIAL = art["DEMO_TRIAL"]
N_CH       = len(EEG_COLS)

print(f"Model loaded  Subject={SUBJECT}  Demo trial={DEMO_TRIAL}")
print(f"T_OPT={T_OPT} ({T_OPT/FS_EFF*1000:.0f} ms)  thresh={thresh:.4f}  FS_EFF={FS_EFF} Hz")

Model loaded  Subject=9  Demo trial=9
T_OPT=12 (194 ms)  thresh=0.5485  FS_EFF=62 Hz


In [3]:
df, _ = load_trial(
    SUBJECT, TACHE, DEMO_TRIAL, DATA_PATH,
    eeg_cols=EEG_COLS, lp=LP, hp=HP,
    decimate=DECIM, car=True, trial_zscore=True,
)
X_raw = df[EEG_COLS].values.astype(np.float32)   # (N_samp, C)
y_mom = df["button"].values.astype(np.int8)        # onset-only press labels
N_samp = len(X_raw)

press_times = np.where(y_mom > 0)[0] / FS_EFF     # seconds
print(f"Trial {DEMO_TRIAL}: {N_samp} samples ({N_samp/FS_EFF:.1f} s)  "
      f"presses={len(press_times)}")

Trial 9: 4298 samples (69.3 s)  presses=33


In [4]:
# ── Feature functions (batch, match eeg_lgb_focused.ipynb exactly) ──────────
def temporal_features(X3d):
    N, T, C = X3d.shape
    X64 = X3d.astype(np.float64)
    tc = np.arange(T, dtype=np.float64) - (T - 1) / 2.0
    tv = (tc ** 2).sum() + 1e-12
    mean_  = X64.mean(axis=1)
    std_   = X64.std(axis=1)
    slope_ = (X64 * tc[None, :, None]).sum(axis=1) / tv
    min_   = X64.min(axis=1)
    argm_  = X64.argmin(axis=1) / T
    rms_   = np.sqrt((X64 ** 2).mean(axis=1))
    return np.stack([mean_, std_, slope_, min_, argm_, rms_], axis=2).reshape(N, C * 6).astype(np.float32)

def freq_features(X3d, fs, batch=2000):
    N, T, C = X3d.shape
    nperseg = min(T, 64)
    out = np.zeros((N, C * 3), dtype=np.float32)
    for s in range(0, N, batch):
        xb = X3d[s:s+batch].transpose(0, 2, 1)
        f, p = welch(xb, fs=fs, nperseg=nperseg, axis=-1)
        df = f[1] - f[0]
        dp = p[:, :, (f >= 0.5) & (f <= 4.0)].sum(axis=-1) * df
        tp = p[:, :, (f >= 4.0) & (f <= 8.0)].sum(axis=-1) * df
        ma = f > 0
        pn = p[:, :, ma] / (p[:, :, ma].sum(-1, keepdims=True) + 1e-12)
        se = -(pn * np.log(pn + 1e-12)).sum(axis=-1)
        out[s:s+batch, 0::3] = dp
        out[s:s+batch, 1::3] = tp
        out[s:s+batch, 2::3] = se
    return out

def hjorth_features(X3d):
    d1 = np.diff(X3d, axis=1)
    d2 = np.diff(d1,  axis=1)
    v0 = X3d.var(axis=1) + 1e-12
    v1 = d1.var(axis=1) + 1e-12
    v2 = d2.var(axis=1) + 1e-12
    mob = np.sqrt(v1 / v0)
    cmp = np.sqrt(v2 / v1) / (mob + 1e-12)
    return np.stack([v0, mob, cmp], axis=2).reshape(len(X3d), -1).astype(np.float32)

def compute_features(X3d, T, fs):
    X_t = X3d[:, -T:, :]
    return np.concatenate(
        [temporal_features(X_t), freq_features(X_t, fs), hjorth_features(X_t)], axis=1
    ).astype(np.float32)

# ── Build all sliding windows via fancy indexing ─────────────────────────────
N_wins = N_samp - T_OPT
print(f"Building {N_wins:,} windows ({N_samp} samples, T_OPT={T_OPT}) …")
idx_mat = np.arange(N_wins)[:, None] + np.arange(T_OPT)[None, :]  # (N_wins, T_OPT)
X_wins = X_raw[idx_mat]   # (N_wins, T_OPT, C)

# Apply EA whitening (same W fitted on training data)
print("Applying Euclidean Alignment …")
X_wins_al = (X_wins @ W.T).astype(np.float32)  # (N_wins, T_OPT, C)
del X_wins

# Compute features in batches
print("Computing features …")
BATCH = 2000
feats = []
for i in range(0, N_wins, BATCH):
    feats.append(compute_features(X_wins_al[i:i+BATCH], T_OPT, FS_EFF))
X_feat = np.concatenate(feats)
del X_wins_al, feats

# Predict and smooth
print("Predicting …")
scores_raw    = model.predict_proba(X_feat)[:, 1].astype(np.float32)
scores_smooth = (pd.Series(scores_raw)
                 .rolling(SMOOTH_WIN, min_periods=1)
                 .mean().values.astype(np.float32))
del X_feat

det_frac = (scores_smooth >= thresh).mean() * 100
print(f"Done. max={scores_smooth.max():.3f}  mean={scores_smooth.mean():.3f}  "
      f"above threshold={det_frac:.1f}%")

Building 4,286 windows (4298 samples, T_OPT=12) …
Applying Euclidean Alignment …
Computing features …
Predicting …
Done. max=0.905  mean=0.276  above threshold=10.5%


/opt/anaconda3/envs/test_env/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [5]:
# ── Animation parameters ─────────────────────────────────────────────────────
STEP       = 4       # raw samples advanced per frame
FPS        = 15      # display frame-rate  → STEP/FS_EFF*FPS ≈ 1× real-time
EEG_WIN_S  = 4.0     # seconds of EEG visible per channel
EEG_WIN    = int(EEG_WIN_S * FS_EFF)
PROB_WIN_S = 30.0    # seconds of probability history visible
PROB_WIN   = int(PROB_WIN_S * FS_EFF)
N_frames   = (N_wins - 1) // STEP

# Precomputed time axes
t_wins    = (T_OPT + np.arange(N_wins)) / FS_EFF      # absolute time per window
t_eeg_rel = np.arange(-EEG_WIN, 0) / FS_EFF            # fixed relative axis for EEG

# EEG y-limits (symmetric around 0, robust to spikes)
eeg_p99  = float(np.percentile(np.abs(X_raw), 99))
eeg_ylim = (-eeg_p99 * 1.25, eeg_p99 * 1.25)

# ── Figure ────────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 13), facecolor=DARK)
gs  = GridSpec(5, 4, figure=fig,
               height_ratios=[1, 1, 1, 1, 1.8],
               hspace=0.38, wspace=0.18)

# 4 × 4 EEG panels
ax_eeg_flat = []
for r in range(4):
    for c in range(4):
        ax = fig.add_subplot(gs[r, c])
        ax.set_facecolor(CARD)
        for sp in ax.spines.values():
            sp.set_color(EDGE)
        ax.tick_params(colors=WHITE, labelsize=5, length=2, pad=1)
        ax.set_xlim(-EEG_WIN_S, 0)
        ax.set_ylim(*eeg_ylim)
        ax.set_title(EEG_COLS[r * 4 + c], color=WHITE, fontsize=7, pad=2)
        if c == 0:
            ax.set_ylabel("µV", color=WHITE, fontsize=5)
        ax_eeg_flat.append(ax)

# Bottom probability panel
ax_prob = fig.add_subplot(gs[4, :])
ax_prob.set_facecolor(CARD)
for sp in ax_prob.spines.values():
    sp.set_color(EDGE)
ax_prob.tick_params(colors=WHITE, labelsize=8)
ax_prob.set_ylim(-0.05, 1.10)
ax_prob.set_ylabel("P(press)", color=WHITE, fontsize=9)
ax_prob.set_xlabel("Time (s)", color=WHITE, fontsize=9)

# Static decoration (plotted once; matplotlib clips to current xlim automatically)
ax_prob.axhline(thresh, color=CORAL, lw=1.5, ls="--", alpha=0.9, zorder=3)
for pt in press_times:
    ax_prob.axvspan(max(0.0, pt - HORIZON / FS_EFF), pt,
                   alpha=0.15, color=GOLD, zorder=1)
    ax_prob.axvline(pt, color=WHITE, lw=0.7, alpha=0.5, zorder=4)

# Legend
leg_handles = [
    mpatches.Patch(color=TEAL, label="P(press)"),
    plt.Line2D([0], [0], color=CORAL, ls="--", lw=1.5, label=f"Threshold ({thresh:.3f})"),
    mpatches.Patch(color=GOLD, alpha=0.4, label=f"Horizon (−{HORIZON/FS_EFF*1000:.0f} ms)"),
    plt.Line2D([0], [0], color=WHITE, lw=0.9, alpha=0.6, label="Button press"),
    plt.Line2D([0], [0], color=CORAL, lw=1.5, label="EEG (detecting)"),
]
ax_prob.legend(handles=leg_handles, loc="upper left",
               facecolor=CARD, labelcolor=WHITE, edgecolor=EDGE,
               fontsize=7, ncol=5)

# Animated artists
lines_eeg  = [ax.plot([], [], lw=0.75, color=TEAL)[0] for ax in ax_eeg_flat]
line_prob, = ax_prob.plot([], [], color=TEAL, lw=1.8, zorder=5)

# Detection indicator (text that flashes coral when above threshold)
det_label = ax_prob.text(
    0.995, 0.94, "\u26a1 DETECTION",
    transform=ax_prob.transAxes,
    color=CORAL, fontsize=9, fontweight="bold",
    ha="right", va="top", alpha=0.0,
)

title_obj = fig.suptitle(
    f"Subject {SUBJECT} \u00b7 Trial {DEMO_TRIAL} \u00b7 Live BCI Prediction",
    color=WHITE, fontsize=11, fontweight="bold", y=0.997,
)

# ── FuncAnimation ─────────────────────────────────────────────────────────────
def init():
    for ln in lines_eeg:
        ln.set_data([], [])
    line_prob.set_data([], [])
    return []

def update(frame):
    idx  = min(T_OPT + frame * STEP, N_samp)   # current sample endpoint
    k_w  = min(frame * STEP, N_wins - 1)        # current window index
    t_cur = idx / FS_EFF
    prob_cur  = float(scores_smooth[k_w])
    detecting = prob_cur >= thresh

    # ── EEG panels: fixed xlim, scrolling data ──────────────────────────────
    i0  = max(0, idx - EEG_WIN)
    n   = idx - i0
    clr = CORAL if detecting else TEAL
    x_rel = t_eeg_rel[-n:]           # relative time ending at 0
    for ch, ln in enumerate(lines_eeg):
        ln.set_data(x_rel, X_raw[i0:idx, ch])
        ln.set_color(clr)

    # ── Probability panel: scrolling xlim ───────────────────────────────────
    i0_w = max(0, k_w - PROB_WIN)
    line_prob.set_data(t_wins[i0_w : k_w + 1], scores_smooth[i0_w : k_w + 1])
    t0_prob = max(0.0, t_cur - PROB_WIN_S)
    ax_prob.set_xlim(t0_prob, t0_prob + PROB_WIN_S)

    # ── Detection label ──────────────────────────────────────────────────────
    det_label.set_alpha(1.0 if detecting else 0.0)

    # ── Title ────────────────────────────────────────────────────────────────
    title_obj.set_text(
        f"Subject {SUBJECT} \u00b7 Trial {DEMO_TRIAL} \u00b7 "
        f"t = {t_cur:.1f} s  |  P = {prob_cur:.3f}"
        + ("  \u26a1" if detecting else "")
    )
    return []

print(f"Rendering {N_frames} frames at {FPS} fps "
      f"({STEP * FPS / FS_EFF:.2f}\u00d7 real-time, "
      f"~{N_frames / FPS:.0f} s video) \u2026")

ani = animation.FuncAnimation(
    fig, update, frames=N_frames, init_func=init,
    interval=1000 // FPS, blit=False,
)

os.makedirs("report", exist_ok=True)
out_path = f"report/live_demo_s{SUBJECT}_t{DEMO_TRIAL}.mp4"
try:
    writer = animation.FFMpegWriter(
        fps=FPS, bitrate=2000,
        extra_args=["-vcodec", "libx264", "-pix_fmt", "yuv420p"],
    )
    ani.save(
        out_path, writer=writer, dpi=80,
        progress_callback=lambda i, n: print(f"\r  frame {i}/{n}", end="", flush=True),
    )
    print(f"\nSaved \u2192 {out_path}")
except Exception as exc:
    out_path = out_path.replace(".mp4", ".gif")
    print(f"FFmpeg unavailable ({exc}), saving GIF \u2026")
    ani.save(out_path, writer="pillow", fps=max(FPS // 2, 5), dpi=60,
             progress_callback=lambda i, n: print(f"\r  frame {i}/{n}", end="", flush=True))
    print(f"\nSaved \u2192 {out_path}")
finally:
    plt.close(fig)

Rendering 1071 frames at 15 fps (0.97× real-time, ~71 s video) …
FFmpeg unavailable ([Errno 2] No such file or directory: 'ffmpeg'), saving GIF …
  frame 1070/1071
Saved → report/live_demo_s9_t9.gif
